In [0]:
from pyspark.sql.functions import *

df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/default/pyspark_files/BigMart Sales.csv"))

display(df)

###DROP (to drop/eliminate column)

In [0]:
df.drop("Item_Visibility", 'Item_Type').display() #Removes one or more columns

### DROP Duplicates->
#####**dropDuplicates** **/** **distinct** **:** Removes redundant rows. `dropDuplicates` can take a `subset` of columns to check for uniqueness.

In [0]:
df.dropDuplicates().display() #remove all duplicates

In [0]:
df.drop_duplicates(subset = ['Item_Type']).display()

In [0]:
df.select('Item_Type').distinct().display() 

## UNION and UNION BY NAME

####Preparing Dataframes

In [0]:
data1 = [('kad', '1'), ('sid', '2')]
schema1 = 'name string, id string'
df1 = spark.createDataFrame(data1, schema1)

data2 = [('3', 'rahul'), ('4', 'jas')]
schema2 = 'id string, name string'
df2 = spark.createDataFrame(data2, schema2)

In [0]:
df1.display()

In [0]:
df2.display()

###Union
#####Merges two DataFrames based on column **position**. If the order of columns differs, the data will be mismatched.


In [0]:
df1.union(df2).display() #Merges two DataFrames based on column **position**. If the order of columns differs, the data will be mismatched.


###Union By Name
##### Merges based on column **names**, making it safer when DataFrames have the same columns in different orders.

In [0]:
df1.unionByName(df2).display()

###String Functions
#####
* **initcap** **:** Capitalises the first letter of each word.
* **lower** **/** **upper** **:** Standardises text case.

In [0]:
df.select(initcap("Item_Type")).display()

In [0]:
df.select(upper("Item_Type").alias("Capitalized Items")).display()

### Date and Time Functions

* **current\_date** **:** Returns the current system date.
* **date\_add** **/** **date\_sub** **:** Adds or subtracts days. (Tip: `date_add` with a negative number acts as a subtraction).
* **datediff** **:** Calculates the days between two date columns.
* **date\_format** **:** Converts a date to a specific string format (e.g., `"dd-MM-yyyy"`).

In [0]:
df3 = df.withColumn('curr_date', current_date())
df3.limit(5).display()

In [0]:
df3 = df3.withColumn('week_after', date_add('curr_date', 7))
df3.limit(5).display()

In [0]:
df3 = df3.withColumn('week_before', date_sub('curr_date', 7))
df3.limit(5).display()

In [0]:
df3.withColumn('week_before', date_add('curr_date', -7)).display()

In [0]:
df3 = df3.withColumn('date_diff', date_diff('curr_date', 'week_before'))
df3.limit(5).display()

In [0]:
df3 = df3.withColumn('formatted_before_date', date_format('week_before', 'dd-MM-yyyy'))
df3.limit(5).display()

## Null Handling Strategies

Missing data is handled via the `df.na` sub-API.

* **dropNa** **:** Removes rows containing nulls. Options include `'any'` (drop if any column is null), `'all'` (drop if all columns are null), or using a `subset`.
* **fillNa** **:** Replaces nulls with a specific value (e.g., "Not Available"). This can also target a `subset` of columns.

In [0]:
df.display()

In [0]:
df.dropna('any').display() #drops all rows with any null values in any column
#df.dropna('all') #drops all rows with null values in all columns

In [0]:
df.dropna(subset=['Outlet_Size']).display() #drops all rows with null values in Outlet_Size column

In [0]:
df.fillna("No Output Available").display() #fills all null values with "No Output Available" 

In [0]:
df.fillna("not available", ["outlet_size"]).display()

## Array and Explode Functions

Useful for columns containing lists or arrays.

* **split** **:** Breaks a string into an array based on a delimiter.
* **Indexing:** Accessing specific elements in an array using `col("array_col")[0]`.
* **explode** **:** Flattens an array so that each element becomes its own row, duplicating the rest of the row data.
* **array\_contains** **:** A boolean check to see if a specific value exists within an array.

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', " ")).display()

In [0]:
df.withColumn('Outlet_Type_Number', split('Outlet_Type', " ")[1]).display() #indexing

In [0]:
df_exp = df.withColumn('Outlet_Type', split('Outlet_Type', " "))

In [0]:
df_exp.select('Outlet_Location_Type', explode(col('Outlet_Type'))).display()

In [0]:
df_exp.withColumn('Type1_flag', array_contains(col('Outlet_Type'), 'Type1')).display()